In [1]:
"""
CLSA Tracking Cohort (Wave 3-4) — Prevalence Analysis
=======================================================
Disease Burden Among Aging Canadians — Individual-Level Corroboration (Section 3.6)

Computes condition prevalence estimates and social participation frequencies
from the CLSA Baseline Stratified Frequencies dataset (participants aged 65+),
used to directly corroborate aggregate GBD 2023 burden patterns.

Input file: clsa_Baseline_StratifiedFrequencies_Amala_Jolly.xlsx
Output:     clsa_prevalence_results.csv
            clsa_social_participation_results.csv

Usage
-----
    python 03_clsa_analysis.py

Place clsa_Baseline_StratifiedFrequencies_Amala_Jolly.xlsx in the same
directory before running.
"""

import os
import pandas as pd

CLSA_FILE = "C:\\Users\\amala\\Downloads\\clsa_Baseline_StratifiedFrequencies Amala Jolly.xlsx"


# ============================================================
# 1. LOAD DATA
# ============================================================

def load_clsa(filepath):
    """Load the CLSA stratified frequency table."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"CLSA file not found: '{filepath}'. "
            f"Place the file in the same directory as this script."
        )
    df = pd.read_excel(filepath)
    print(f"[OK] Loaded CLSA file: {filepath}")
    print(f"     Shape: {df.shape}")
    print(f"     Variables: {sorted(df['variable'].unique().tolist())}")
    return df


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def get_yes_prevalence(df, variable, label='Yes'):
    """Compute prevalence (%) for a Yes/No variable, excluding 'DO NOT READ'
    and 'Refused' responses from both numerator and denominator."""
    sub = df[df['variable'] == variable]
    yes_n = sub[sub['label:en'] == label]['Overall Frequency'].sum()
    denom = sub[
        ~sub['label:en'].str.contains('DO NOT READ|Refused', na=False)
    ]['Overall Frequency'].sum()
    pct = yes_n / denom * 100 if denom > 0 else 0.0
    return int(yes_n), int(denom), round(pct, 1)


def get_total_sample(df):
    """Use GEN_HLTH_TRM as the anchor variable for total sample size,
    since it is the most complete variable in the dataset."""
    gen = df[df['variable'] == 'GEN_HLTH_TRM']
    valid = gen[~gen['label:en'].str.contains('DO NOT READ|Refused', na=False)]
    total = int(valid['Overall Frequency'].sum())
    breakdown = {
        'Males 65-74':   int(valid['M [65-74 (3)]'].sum()),
        'Females 65-74': int(valid['F [65-74 (3)]'].sum()),
        'Males 75+':     int(valid['M [75+ (4)]'].sum()),
        'Females 75+':   int(valid['F [75+ (4)]'].sum()),
    }
    return total, breakdown


# ============================================================
# 3. CHRONIC CONDITION PREVALENCES
# ============================================================

def compute_condition_prevalences(df):
    """Compute Yes/No prevalence for all chronic condition variables.
    Returns a DataFrame with columns: Condition, Variable, N_Yes, N_Total, Prevalence_Pct,
    GBD_Category_Match."""
    conditions = [
        ('Hypertension',             'CCT_HBP_TRM',   'Cardiovascular diseases'),
        ('Heart disease',            'CCT_HEART_TRM', 'Cardiovascular diseases'),
        ('Stroke',                   'CCT_CVA_TRM',   'Neurological disorders'),
        ('Back problems',            'CCT_BCKP_TRM',  'Musculoskeletal disorders'),
        ('Osteoarthritis of knee',   'CCT_OAKNEE_TRM','Musculoskeletal disorders'),
        ('Osteoarthritis of hip',    'CCT_OAHIP_TRM', 'Musculoskeletal disorders'),
        ('Rheumatoid arthritis',     'CCT_RA_TRM',    'Musculoskeletal disorders'),
        ('Osteoporosis',             'CCT_OSTPO_TRM', 'Musculoskeletal disorders'),
        ('Diabetes',                 'CCT_DIAB_TRM',  'Diabetes and kidney diseases'),
        ('Asthma',                   'CCT_ASTHM_TRM', 'Chronic respiratory diseases'),
        ('COPD',                     'CCT_COPD_TRM',  'Chronic respiratory diseases'),
        ('Anxiety',                  'CCT_ANXI_TRM',  'Mental disorders'),
        ('Alzheimer\'s disease',     'CCT_ALZH_TRM',  'Neurological disorders'),
        ('Parkinson\'s disease',     'PKD_PARK_MCQ',  'Neurological disorders'),
    ]

    rows = []
    for condition, variable, gbd_cat in conditions:
        yes_n, denom, pct = get_yes_prevalence(df, variable)
        rows.append({
            'Condition': condition,
            'Variable': variable,
            'N_Yes': yes_n,
            'N_Total': denom,
            'Prevalence_Pct': pct,
            'GBD_Category_Match': gbd_cat,
        })

    return pd.DataFrame(rows).sort_values('Prevalence_Pct', ascending=False).reset_index(drop=True)


# ============================================================
# 4. DEPRESSION SCREENING
# ============================================================

def compute_depression_screen(df):
    """Compute depression screening result: positive vs negative screen."""
    dep = df[df['variable'] == 'DEP_DPSFD_TRM']
    positive = int(dep[dep['label:en'].str.contains('Positive', na=False)]['Overall Frequency'].sum())
    negative = int(dep[dep['label:en'].str.contains('Negative', na=False)]['Overall Frequency'].sum())
    denom = positive + negative  # exclude inconclusive
    pct = positive / denom * 100 if denom > 0 else 0.0
    return positive, denom, round(pct, 1)


# ============================================================
# 5. SOCIAL PARTICIPATION AND MENTAL WELLBEING
# ============================================================

def compute_social_participation(df):
    """Compute social participation and loneliness frequencies."""
    results = {}

    # Volunteering — at least monthly (daily + weekly + monthly)
    vol = df[df['variable'] == 'SPA_VOLUN_TRM']
    monthly_plus = int(vol[vol['label:en'].isin([
        'At least once a day', 'At least once a week', 'At least once a month'
    ])]['Overall Frequency'].sum())
    denom_vol = int(vol[~vol['label:en'].str.contains('DO NOT READ|Refused', na=False)]['Overall Frequency'].sum())
    results['Volunteering (at least monthly)'] = (monthly_plus, denom_vol, round(monthly_plus / denom_vol * 100, 1))

    # Other activities — at least monthly
    act = df[df['variable'] == 'SPA_OTACT_TRM']
    act_monthly = int(act[act['label:en'].isin([
        'At least once a day', 'At least once a week', 'At least once a month'
    ])]['Overall Frequency'].sum())
    denom_act = int(act[~act['label:en'].str.contains('DO NOT READ|Refused', na=False)]['Overall Frequency'].sum())
    results['Other recreational activities (at least monthly)'] = (act_monthly, denom_act, round(act_monthly / denom_act * 100, 1))

    # Loneliness — agree or strongly agree
    lon = df[df['variable'] == 'ENV_FLLNLY_MCQ']
    agree = int(lon[lon['label:en'].isin(['Agree', 'Strongly agree'])]['Overall Frequency'].sum())
    denom_lon = int(lon[~lon['label:en'].isin(["Don't Know/No Answer", 'Refused', 'Missing'])]['Overall Frequency'].sum())
    results['Loneliness (agree or strongly agree)'] = (agree, denom_lon, round(agree / denom_lon * 100, 1))

    return results


# ============================================================
# 6. SELF-RATED HEALTH
# ============================================================

def compute_selfrated_health(df):
    """Compute self-rated general health: good, very good, or excellent."""
    gen = df[df['variable'] == 'GEN_HLTH_TRM']
    good_plus = int(gen[gen['label:en'].isin(['Excellent', 'Very good', 'Good'])]['Overall Frequency'].sum())
    denom = int(gen[~gen['label:en'].str.contains('DO NOT READ|Refused', na=False)]['Overall Frequency'].sum())
    pct = good_plus / denom * 100 if denom > 0 else 0.0
    return good_plus, denom, round(pct, 1)


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print("CLSA Tracking Cohort (Wave 3-4) — Prevalence Analysis")
    print("For: Disease Burden Among Aging Canadians, Section 3.6")
    print("=" * 70)
    print()

    df = load_clsa(CLSA_FILE)

    # ── Sample size ──
    total, breakdown = get_total_sample(df)
    print(f"\n{'='*70}")
    print("SAMPLE SIZE")
    print(f"{'='*70}")
    print(f"Total participants (aged 65+): {total:,}")
    for group, n in breakdown.items():
        print(f"  {group:20s}: {n:,}  ({n/total*100:.1f}%)")

    # ── Condition prevalences ──
    print(f"\n{'='*70}")
    print("CHRONIC CONDITION PREVALENCES (sorted by prevalence)")
    print(f"{'='*70}")
    prev_df = compute_condition_prevalences(df)
    print(prev_df.to_string(index=False))

    # ── Depression screening ──
    dep_pos, dep_denom, dep_pct = compute_depression_screen(df)
    print(f"\n{'='*70}")
    print("DEPRESSION SCREENING (DEP_DPSFD_TRM)")
    print(f"{'='*70}")
    print(f"Positive screen: {dep_pos:,} / {dep_denom:,} = {dep_pct}%")
    print(f"(Inconclusive results excluded from denominator)")

    # ── Self-rated health ──
    good_n, good_denom, good_pct = compute_selfrated_health(df)
    print(f"\n{'='*70}")
    print("SELF-RATED GENERAL HEALTH")
    print(f"{'='*70}")
    print(f"Good, Very good, or Excellent: {good_n:,} / {good_denom:,} = {good_pct}%")

    # ── Social participation ──
    print(f"\n{'='*70}")
    print("SOCIAL PARTICIPATION AND LONELINESS")
    print(f"{'='*70}")
    social = compute_social_participation(df)
    for label, (n, denom, pct) in social.items():
        print(f"  {label}")
        print(f"    {n:,} / {denom:,} = {pct}%")

    # ── Key findings for paper ──
    print(f"\n{'='*70}")
    print("KEY FINDINGS FOR PAPER (Section 3.6)")
    print(f"{'='*70}")
    print(f"Sample: {total:,} CLSA participants aged 65+")
    print(f"Hypertension:          {prev_df[prev_df['Condition']=='Hypertension']['Prevalence_Pct'].values[0]}%  → corroborates GBD cardiovascular burden (#2)")
    print(f"Back problems:         {prev_df[prev_df['Condition']=='Back problems']['Prevalence_Pct'].values[0]}%  → corroborates GBD musculoskeletal burden (#4)")
    print(f"OA knee:               {prev_df[prev_df['Condition']=='Osteoarthritis of knee']['Prevalence_Pct'].values[0]}%  → corroborates GBD musculoskeletal burden (#4)")
    print(f"Diabetes:              {prev_df[prev_df['Condition']=='Diabetes']['Prevalence_Pct'].values[0]}%  → corroborates GBD diabetes & kidney burden (#6)")
    print(f"Heart disease:         {prev_df[prev_df['Condition']=='Heart disease']['Prevalence_Pct'].values[0]}%  → corroborates GBD cardiovascular burden (#2)")
    print(f"COPD:                  {prev_df[prev_df['Condition']=='COPD']['Prevalence_Pct'].values[0]}%   → corroborates CIHI #1 hospitalization cause")
    print(f"Depression (screen):   {dep_pct}%  → corroborates GBD mental disorders burden (#10 by size, fastest growing)")
    print(f"Anxiety:               {prev_df[prev_df['Condition']=='Anxiety']['Prevalence_Pct'].values[0]}%   → corroborates GBD mental disorders burden")
    vol_pct = social['Volunteering (at least monthly)'][2]
    act_pct = social['Other recreational activities (at least monthly)'][2]
    lon_pct = social['Loneliness (agree or strongly agree)'][2]
    print(f"Volunteering monthly+: {vol_pct}%  → social participation protective factor")
    print(f"Other activities m+:   {act_pct}%  → social participation protective factor")
    print(f"Loneliness:            {lon_pct}%   → social determinant of mental health")
    print(f"Self-rated health 'good or better': {good_pct}%")

    # ── Export ──
    prev_df.to_csv('clsa_prevalence_results.csv', index=False)
    social_rows = [
        {'Indicator': label, 'N': n, 'N_Total': denom, 'Percentage': pct}
        for label, (n, denom, pct) in social.items()
    ]
    social_rows.append({'Indicator': 'Depression positive screen', 'N': dep_pos, 'N_Total': dep_denom, 'Percentage': dep_pct})
    social_rows.append({'Indicator': 'Self-rated health good or better', 'N': good_n, 'N_Total': good_denom, 'Percentage': good_pct})
    pd.DataFrame(social_rows).to_csv('clsa_social_participation_results.csv', index=False)

    print(f"\n[OK] Exported: clsa_prevalence_results.csv, clsa_social_participation_results.csv")
    print("=" * 70)


if __name__ == '__main__':
    main()

CLSA Tracking Cohort (Wave 3-4) — Prevalence Analysis
For: Disease Burden Among Aging Canadians, Section 3.6

[OK] Loaded CLSA file: C:\Users\amala\Downloads\clsa_Baseline_StratifiedFrequencies Amala Jolly.xlsx
     Shape: (87, 9)
     Variables: ['CCT_ALZH_TRM', 'CCT_ANXI_TRM', 'CCT_ASTHM_TRM', 'CCT_BCKP_TRM', 'CCT_COPD_TRM', 'CCT_CVA_TRM', 'CCT_DIAB_TRM', 'CCT_HBP_TRM', 'CCT_HEART_TRM', 'CCT_OAHIP_TRM', 'CCT_OAKNEE_TRM', 'CCT_OSTPO_TRM', 'CCT_RA_TRM', 'DEP_DPSFD_TRM', 'ENV_FLLNLY_MCQ', 'GEN_HLTH_TRM', 'PKD_PARK_MCQ', 'SPA_OTACT_TRM', 'SPA_VOLUN_TRM']

SAMPLE SIZE
Total participants (aged 65+): 21,220
  Males 65-74         : 2,292  (10.8%)
  Females 65-74       : 2,336  (11.0%)
  Males 75+           : 2,102  (9.9%)
  Females 75+         : 2,100  (9.9%)

CHRONIC CONDITION PREVALENCES (sorted by prevalence)
             Condition       Variable  N_Yes  N_Total  Prevalence_Pct           GBD_Category_Match
          Hypertension    CCT_HBP_TRM   8102    21213            38.2      Cardiova